In [ ]:
import re
import sys
from dataclasses import dataclass
from pathlib import Path

from PyQt6.QtCore import Qt, QTimer
from PyQt6.QtWidgets import (
    QApplication, QFileDialog, QLabel, QPushButton, QSlider, QHBoxLayout,
    QVBoxLayout, QWidget, QMessageBox, QComboBox
)

from ebooklib import epub, ITEM_DOCUMENT
from bs4 import BeautifulSoup

PARA_TOKEN = "\n"

DEFAULT_EPUB_PATH = Path(
    "/mnt/data/[First Law Book 1] Joe Abercrombie - First Law 1 The Blade Itself "
    "(2007, St. Martin's Minotaur) - libgen.li.epub"
)


def epub_to_tokens(epub_path: str) -> list[str]:
    book = epub.read_epub(epub_path)
    id_to_item = {item.get_id(): item for item in book.get_items()}

    tokens: list[str] = []
    for item_id, _linear in book.spine:
        item = id_to_item.get(item_id)
        if not item or item.get_type() != ITEM_DOCUMENT:
            continue

        soup = BeautifulSoup(item.get_content(), "lxml")
        for tag in soup(["script", "style", "nav", "header", "footer", "noscript"]):
            tag.decompose()

        text = soup.get_text("\n").replace("\r", "")
        paragraphs = [p.strip() for p in re.split(r"\n\s*\n+", text) if p.strip()]

        for p in paragraphs:
            p = re.sub(r"\s+", " ", p).strip()
            if not p:
                continue
            tokens.extend(p.split(" "))
            tokens.append(PARA_TOKEN)

    while tokens and tokens[-1] == PARA_TOKEN:
        tokens.pop()
    return tokens


def orp_index(word: str) -> int:
    n = len(word)
    if n <= 1:
        return 0
    if n <= 5:
        return 1
    if n <= 9:
        return 2
    if n <= 13:
        return 3
    return max(0, int(n * 0.35))


@dataclass
class PauseFactors:
    sentence: float = 1.8
    comma: float = 1.35
    paragraph: float = 2.2


THEMES = {
    "Dark": {
        "bg": "#0b0d10",
        "fg": "#e9eef5",
        "muted": "#a9b4c2",
        "panel": "#121826",
        "button_bg": "#1b2433",
        "button_border": "#2b3a52",
        "orp": "#ff6b81",   # softer than hot pink
    },
    "Sepia": {
        "bg": "#f4ecd8",
        "fg": "#2b2418",
        "muted": "#5a4d3a",
        "panel": "#efe3c6",
        "button_bg": "#e7d7b5",
        "button_border": "#c8b489",
        "orp": "#b23a48",   # muted red
    },
    "Light": {
        "bg": "#ffffff",
        "fg": "#111827",
        "muted": "#4b5563",
        "panel": "#f3f4f6",
        "button_bg": "#e5e7eb",
        "button_border": "#cbd5e1",
        "orp": "#d946ef",   # soft purple accent
    }
}


class SpeedReader(QWidget):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("EPUB Speed Reader (Python)")
        self.resize(900, 520)

        self.tokens: list[str] = []
        self.idx: int = 0
        self.playing: bool = False
        self.pause = PauseFactors()

        self.theme_name = "Sepia"
        self.font_size = 64

        # Display
        self.word_label = QLabel("Loading default EPUB…" if DEFAULT_EPUB_PATH.exists() else "Load an EPUB to start")
        self.word_label.setAlignment(Qt.AlignmentFlag.AlignCenter)

        self.progress_label = QLabel("")
        self.progress_label.setAlignment(Qt.AlignmentFlag.AlignCenter)

        # Controls
        self.load_btn = QPushButton("Load EPUB")
        self.play_btn = QPushButton("Play")
        self.back_btn = QPushButton("⟲ -10")
        self.fwd_btn = QPushButton("+10 ⟳")

        self.wpm_slider = QSlider(Qt.Orientation.Horizontal)
        self.wpm_slider.setMinimum(150)
        self.wpm_slider.setMaximum(900)
        self.wpm_slider.setSingleStep(10)
        self.wpm_slider.setValue(400)
        self.wpm_label = QLabel("WPM: 400")

        # ✅ Theme selector
        self.theme_box = QComboBox()
        self.theme_box.addItems(list(THEMES.keys()))
        self.theme_box.setCurrentText(self.theme_name)

        # ✅ Font size slider
        self.font_slider = QSlider(Qt.Orientation.Horizontal)
        self.font_slider.setMinimum(36)
        self.font_slider.setMaximum(96)
        self.font_slider.setSingleStep(2)
        self.font_slider.setValue(self.font_size)
        self.font_label = QLabel(f"Size: {self.font_size}")

        # Layouts
        top = QHBoxLayout()
        top.addWidget(self.load_btn)
        top.addStretch(1)
        top.addWidget(self.back_btn)
        top.addWidget(self.play_btn)
        top.addWidget(self.fwd_btn)
        top.addStretch(1)

        top.addWidget(QLabel("Theme:"))
        top.addWidget(self.theme_box)

        top.addSpacing(10)
        top.addWidget(self.wpm_label)
        top.addWidget(self.wpm_slider)

        top.addSpacing(10)
        top.addWidget(self.font_label)
        top.addWidget(self.font_slider)

        center = QVBoxLayout()
        center.addStretch(1)
        center.addWidget(self.word_label)
        center.addSpacing(10)
        center.addWidget(self.progress_label)
        center.addStretch(1)

        root = QVBoxLayout()
        root.addLayout(top)
        root.addLayout(center)
        self.setLayout(root)

        # Timer
        self.timer = QTimer(self)
        self.timer.setSingleShot(True)
        self.timer.timeout.connect(self._tick)

        # Wiring
        self.load_btn.clicked.connect(self.load_epub_dialog)
        self.play_btn.clicked.connect(self.toggle_play)
        self.back_btn.clicked.connect(lambda: self.jump(-10))
        self.fwd_btn.clicked.connect(lambda: self.jump(10))

        self.wpm_slider.valueChanged.connect(lambda v: self.wpm_label.setText(f"WPM: {v}"))

        self.theme_box.currentTextChanged.connect(self.set_theme)
        self.font_slider.valueChanged.connect(self.set_font_size)

        self.setFocusPolicy(Qt.FocusPolicy.StrongFocus)

        # Apply initial theme/font
        self.apply_theme_styles()
        self.apply_font_styles()

        # Auto-load default
        self.auto_load_default()

    def set_theme(self, name: str):
        self.theme_name = name
        self.apply_theme_styles()
        self._render_current()

    def set_font_size(self, size: int):
        self.font_size = size
        self.font_label.setText(f"Size: {size}")
        self.apply_font_styles()
        self._render_current()

    def apply_theme_styles(self):
        t = THEMES[self.theme_name]
        # Window background + default text
        self.setStyleSheet(f"""
            QWidget {{
                background: {t['bg']};
                color: {t['fg']};
                font-family: system-ui, -apple-system, Segoe UI, Roboto, Arial;
            }}
            QPushButton {{
                background: {t['button_bg']};
                border: 1px solid {t['button_border']};
                padding: 8px 10px;
                border-radius: 10px;
            }}
            QPushButton:hover {{
                opacity: 0.95;
            }}
            QComboBox {{
                background: {t['panel']};
                border: 1px solid {t['button_border']};
                padding: 6px 8px;
                border-radius: 10px;
            }}
            QSlider::groove:horizontal {{
                height: 6px;
                background: {t['panel']};
                border-radius: 3px;
            }}
            QSlider::handle:horizontal {{
                width: 14px;
                margin: -6px 0;
                border-radius: 7px;
                background: {t['button_bg']};
                border: 1px solid {t['button_border']};
            }}
        """)

        # Muted progress label
        self.progress_label.setStyleSheet(f"font-size: 14px; color: {t['muted']};")

    def apply_font_styles(self):
        self.word_label.setStyleSheet(f"font-size: {self.font_size}px; font-weight: 750;")

    def keyPressEvent(self, event):
        if event.key() == Qt.Key.Key_Space:
            self.toggle_play()
            return
        if event.key() == Qt.Key.Key_Left:
            self.jump(-10)
            return
        if event.key() == Qt.Key.Key_Right:
            self.jump(10)
            return
        if event.key() == Qt.Key.Key_Up:
            self.wpm_slider.setValue(min(self.wpm_slider.maximum(), self.wpm_slider.value() + 10))
            return
        if event.key() == Qt.Key.Key_Down:
            self.wpm_slider.setValue(max(self.wpm_slider.minimum(), self.wpm_slider.value() - 10))
            return
        super().keyPressEvent(event)

    def auto_load_default(self):
        if DEFAULT_EPUB_PATH.exists():
            self.load_epub_path(str(DEFAULT_EPUB_PATH))
        else:
            self.word_label.setText("Default EPUB path not found. Click “Load EPUB”.")

    def load_epub_dialog(self):
        path, _ = QFileDialog.getOpenFileName(self, "Open EPUB", "", "EPUB Files (*.epub)")
        if path:
            self.load_epub_path(path)

    def load_epub_path(self, path: str):
        self.stop()
        self.tokens = []
        self.idx = 0
        self.word_label.setText("Loading…")
        QApplication.processEvents()

        try:
            self.tokens = epub_to_tokens(path)
        except Exception as e:
            QMessageBox.critical(self, "Error", f"Failed to load EPUB:\n{e}")
            self.tokens = []
            self.word_label.setText("Load an EPUB to start")
            return

        if not self.tokens:
            QMessageBox.warning(self, "No Text Found", "Could not extract readable text from this EPUB.")
            self.word_label.setText("Load an EPUB to start")
            return

        self.idx = 0
        self._render_current()
        self._update_progress()

    def toggle_play(self):
        if not self.tokens:
            return
        self.playing = not self.playing
        self.play_btn.setText("Pause" if self.playing else "Play")
        if self.playing:
            self._schedule_next()
        else:
            self.timer.stop()

    def stop(self):
        self.playing = False
        self.play_btn.setText("Play")
        self.timer.stop()

    def jump(self, delta: int):
        if not self.tokens:
            return
        self.idx = max(0, min(len(self.tokens) - 1, self.idx + delta))
        self._render_current()
        self._update_progress()

    def _update_progress(self):
        if self.tokens:
            self.progress_label.setText(f"{self.idx + 1:,} / {len(self.tokens):,}")
        else:
            self.progress_label.setText("")

    def _delay_ms_for_token(self, tok: str) -> int:
        wpm = self.wpm_slider.value()
        base = 60000 / max(1, wpm)
        if tok == PARA_TOKEN:
            return int(base * self.pause.paragraph)
        if re.search(r'[.?!]["\')\]]?$', tok):
            return int(base * self.pause.sentence)
        if re.search(r'[,;:]["\')\]]?$', tok):
            return int(base * self.pause.comma)
        return int(base)

    def _render_current(self):
        if not self.tokens:
            return
        tok = self.tokens[self.idx]
        t = THEMES[self.theme_name]

        if tok == PARA_TOKEN:
            self.word_label.setText("⏎")
            # reduce opacity via style
            self.word_label.setStyleSheet(f"font-size: {self.font_size}px; font-weight: 750; color: {t['muted']};")
            return

        i = orp_index(tok)
        left, mid, right = tok[:i], tok[i:i+1], tok[i+1:]
        self.word_label.setTextFormat(Qt.TextFormat.RichText)
        self.word_label.setStyleSheet(f"font-size: {self.font_size}px; font-weight: 750; color: {t['fg']};")
        self.word_label.setText(
            f"{self._esc(left)}"
            f"<span style='color:{t['orp']}'>{self._esc(mid)}</span>"
            f"{self._esc(right)}"
        )

    def _esc(self, s: str) -> str:
        return (
            s.replace("&", "&amp;")
             .replace("<", "&lt;")
             .replace(">", "&gt;")
             .replace('"', "&quot;")
             .replace("'", "&#039;")
        )

    def _tick(self):
        if not self.playing or not self.tokens:
            return
        if self.idx >= len(self.tokens) - 1:
            self.stop()
            return
        self.idx += 1
        self._render_current()
        self._update_progress()
        self._schedule_next()

    def _schedule_next(self):
        if not self.playing or not self.tokens:
            return
        self.timer.start(self._delay_ms_for_token(self.tokens[self.idx]))


def main():
    app = QApplication(sys.argv)
    w = SpeedReader()
    w.show()
    sys.exit(app.exec())


if __name__ == "__main__":
    main()

qt.qpa.fonts: Populating font family aliases took 169 ms. Replace uses of missing font family "System-ui" with one that exists to avoid this cost. 
2026-03-06 20:15:12.506 Python[16774:1056725] The class 'NSOpenPanel' overrides the method identifier.  This method is implemented by class 'NSWindow'
/var/folders/v2/yl28_x4s6k9dk8gy5c271hkr0000gn/T/ipykernel_16774/3967726777.py:33: XMLParsedAsHTMLWarning: It looks like you're parsing an XML document using an HTML parser. If this really is an HTML document (maybe it's XHTML?), you can ignore or filter this warning. If it's XML, you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the lxml package installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.
  soup = BeautifulSoup(item.get_content(), "lxml")
